<a href="https://colab.research.google.com/github/arshenoym25-vjti/Sunrise-AMC-Assignment/blob/main/Sunrise_AMC__Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Sunrise AMC Assignment


*   Name - Adithya Ramesh Shenoy
*   Branch - MTECH (AI & DS)

In [1]:
# ==========================================
# CELL 1: Setup & Initialization
# ==========================================
from google.colab import drive
import os
import subprocess
import time

# 1. Mount Drive
print("Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)
BASE_PATH = "/content/drive/MyDrive/Projects/HDFC AMC Project"

# Create directories
os.makedirs(f"{BASE_PATH}/data", exist_ok=True)
os.makedirs(f"{BASE_PATH}/output", exist_ok=True)
os.makedirs(f"{BASE_PATH}/input", exist_ok=True)
print(f"Project directories ready at: {BASE_PATH}")

# 2. System Dependencies
print("\nInstalling system dependencies (zstd)...")
!sudo apt-get update -qq && sudo apt-get install -y zstd curl -qq

# 3. Python Libraries
print("\nInstalling open-source ML libraries...")
!pip install -q faster-whisper chromadb sentence-transformers langchain langchain-community langchain-text-splitters pypdf requests

# 4. Ollama Local LLM Setup
print("\nInstalling and starting Ollama background server...")
!curl -fsSL https://ollama.com/install.sh | sh

# Start server silently in the background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5) # Wait for server to boot

print("\nPulling Llama 3 weights (this takes ~1-2 minutes)...")
!ollama pull llama3

print("\n✅ Environment Ready! Please ensure your PDF and MP3 are in the /input folder, then proceed to Cell 2.")

Mounting Google Drive...
Mounted at /content/drive
Project directories ready at: /content/drive/MyDrive/Projects/HDFC AMC Project

Installing system dependencies (zstd)...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80), connection timed out
W: Failed to fetch https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443:
W: Failed to fetch https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443:
W: Some index files failed to download. They have been ignored, or old ones used instead.
debconf: unable to initialize frontend: Dialog
debconf: (No us

In [2]:
# ==========================================
# CELL 2: Core Functions & Edge Case Handling
# ==========================================
import json
import requests
from faster_whisper import WhisperModel
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# --- PART 1: Voice Transcription ---
def transcribe_audio(audio_path, output_path):
    print("\n[1/4] Loading Faster-Whisper model...")
    model = WhisperModel("small", device="cuda", compute_type="float16")

    print(f"      Transcribing {audio_path.split('/')[-1]}...")
    segments, info = model.transcribe(audio_path, word_timestamps=True)

    transcript_text = ""
    words_data = []

    for segment in segments:
        transcript_text += segment.text + " "
        for word in segment.words:
            words_data.append({
                "word": word.word, "start": word.start, "end": word.end, "probability": word.probability
            })

    transcript_text = transcript_text.strip()

    # PROACTIVE EDGE CASE: Empty Audio
    if not transcript_text:
        print("\n[WARNING] No speech detected in the audio file.")
        return {"transcript_text": "", "confidence_scores": []}

    output_json = {"transcript_text": transcript_text, "confidence_scores": words_data}

    with open(output_path, "w") as f:
        json.dump(output_json, f, indent=4)

    return output_json

# --- PART 2: RAG Ingestion ---
def build_rag_engine(pdf_path, db_path):
    print("\n[2/4] Loading and chunking PDF...")
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500, chunk_overlap=50, separators=["\nQ", "\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.split_documents(documents)

    print("      Initializing embedding model and ChromaDB...")
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vector_store = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_path)
    return vector_store

# --- PART 3: LLM Generation ---
def query_local_llm(context, user_query):
    url = "http://localhost:11434/api/generate"

    # PROACTIVE EDGE CASE: Out of Scope Constraint
    prompt = f"""You are a helpful Investor Support Assistant for Sunrise Asset Management.
    Use the following FAQ context to answer the user's query.
    You MUST cite the specific FAQ question number (e.g., Q1, Q4) as your source in your answer.
    If the answer is NOT explicitly contained in the context, you must politely state that you do not know and refuse to answer. Do not guess.

    Context:
    {context}

    User Query: {user_query}

    Answer:"""

    payload = {"model": "llama3", "prompt": prompt, "stream": False}
    response = requests.post(url, json=payload)
    return response.json().get("response", "")

In [3]:
# ==========================================
# CELL 3: Pipeline Execution
# ==========================================
def run_pipeline():
    audio_path = f"{BASE_PATH}/input/investor_sample.mp3"
    pdf_path = f"{BASE_PATH}/input/SunriseAMC_FAQ.pdf"
    output_path = f"{BASE_PATH}/output/transcript.json"
    db_path = f"{BASE_PATH}/data/chroma_db"

    print("==========================================")
    print("   Starting Voice-to-RAG Pipeline         ")
    print("==========================================")

    transcript_data = transcribe_audio(audio_path, output_path)
    query_text = transcript_data["transcript_text"]

    if not query_text:
        return print("\nPipeline Halted: Audio input was empty.")

    print(f"\n[Transcribed Query]: '{query_text}'\n")

    vector_store = build_rag_engine(pdf_path, db_path)

    print("\n[3/4] Retrieving relevant context...")
    docs = vector_store.similarity_search(query_text, k=2)
    context = "\n".join([doc.page_content for doc in docs])

    print("\n[4/4] Generating grounded response via Llama 3...")

    # LATENCY TRACKING
    start_time = time.time()
    final_answer = query_local_llm(context, query_text)
    latency = time.time() - start_time

    print("\n==========================================")
    print("             FINAL RESPONSE               ")
    print("==========================================")
    print(final_answer)
    print(f"==========================================\n[Inference Latency: {latency:.2f} seconds]")

run_pipeline()

print("\n--- Verifying Structured JSON Output ---")
with open(f"{BASE_PATH}/output/transcript.json", "r") as f:
    saved_json = json.load(f)
print(json.dumps(saved_json, indent=2)[:400] + "\n... [truncated]")

   Starting Voice-to-RAG Pipeline         

[1/4] Loading Faster-Whisper model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


      Transcribing investor_sample.mp3...

[Transcribed Query]: 'Hi, I recently invested in an equity mutual fund through my SIP.  I want to know, if I redeem my units after 14 months, how will my gains be taxed?  And will any TDS be deducted?'


[2/4] Loading and chunking PDF...
      Initializing embedding model and ChromaDB...


/tmp/ipykernel_3744/3788848869.py:56: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


[3/4] Retrieving relevant context...

[4/4] Generating grounded response via Llama 3...

             FINAL RESPONSE               
Hello! Thank you for your query.

According to our FAQ (Q9), Long-Term Capital Gains (LTCG) are taxed at 12.5% on gains exceeding Rs. 1.25 lakh in a financial year, and your units have been held for more than 12 months (14 months in this case). Therefore, your gains would be subject to LTCG tax rate of 12.5%.

Regarding TDS, according to our FAQ (Q10), no TDS will be deducted on mutual fund redemptions for resident Indian investors like yourself. So, there won't be any TDS deducted on the redemption.

Please note that tax laws are subject to change, and it's always a good idea to consult your tax advisor for personalized advice.

Best regards,
Sunrise Asset Management Co. Ltd.
[Inference Latency: 122.56 seconds]

--- Verifying Structured JSON Output ---
{
  "transcript_text": "Hi, I recently invested in an equity mutual fund through my SIP.  I want to kno

In [4]:
# ==========================================
# CELL 4: Quality & Benchmark Eval
# ==========================================
def run_manual_evaluation():
    db_path = f"{BASE_PATH}/data/chroma_db"
    print("\n" + "="*50)
    print(" 🧪 INITIATING PIPELINE EVALUATION & BENCHMARKS ")
    print("="*50 + "\n")

    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vector_store = Chroma(persist_directory=db_path, embedding_function=embeddings)

    test_cases = [
        {"type": "Standard Query", "query": "How long does a liquid fund redemption take?", "expected_citation": "Q7"},
        {"type": "Edge Case: Strict Rule", "query": "Can I invest in mutual funds for my 10-year-old son?", "expected_citation": "Q3"},
        {"type": "Edge Case: Out of Scope", "query": "What are the latest RBI interest rates?", "expected_citation": None}
    ]

    total_latency = 0

    for i, test in enumerate(test_cases):
        print(f"Test {i+1}: {test['type']}")
        print(f"Query: '{test['query']}'")

        start_time = time.time()
        docs = vector_store.similarity_search(test["query"], k=2)
        context = "\n".join([doc.page_content for doc in docs])
        answer = query_local_llm(context, test["query"])
        latency = time.time() - start_time
        total_latency += latency

        print(f"Response: {answer}")
        print(f"⏱️ Latency: {latency:.2f}s")

        if test["expected_citation"]:
            if test["expected_citation"] in answer:
                print(f"✅ Pass: Correct Source Cited ({test['expected_citation']})")
            else:
                print(f"❌ Fail: Missing Source Citation ({test['expected_citation']})")
        else:
            print("✅ Pass: Handled Out-of-Scope gracefully")
        print("-" * 50)

    print(f"\n📊 Average LLM Generation Latency: {(total_latency / len(test_cases)):.2f} seconds")

run_manual_evaluation()


 🧪 INITIATING PIPELINE EVALUATION & BENCHMARKS 



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_3744/2433527773.py:11: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(persist_directory=db_path, embedding_function=embeddings)


Test 1: Standard Query
Query: 'How long does a liquid fund redemption take?'
Response: According to FAQ Q7, the answer is: "Liquid and Overnight funds — T+1 business day." This means that a liquid fund redemption request typically takes one trading day (T+1) to process.
⏱️ Latency: 1.70s
✅ Pass: Correct Source Cited (Q7)
--------------------------------------------------
Test 2: Edge Case: Strict Rule
Query: 'Can I invest in mutual funds for my 10-year-old son?'
Response: According to our FAQ context (Q3), a minor can indeed invest in mutual funds through a parent or legal guardian as the first holder. The folio will be held in the minor's name, and once they turn 18, the account must be converted to a major account before any further transactions are processed.

So, yes, you can invest in mutual funds for your 10-year-old son (Q3).
⏱️ Latency: 2.54s
✅ Pass: Correct Source Cited (Q3)
--------------------------------------------------
Test 3: Edge Case: Out of Scope
Query: 'What are the